In [1]:
!pip install mlflow dagshub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 684.1 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 36.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## Load Champion Model from DagsHub Registry
We connect to DagsHub and load the best registered pipeline directly from the Model Registry. No manual preprocessing is needed because the pipeline includes all cleaning and feature engineering steps internally.

In [2]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import os

os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML-Assignment-2')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow")
print("DagsHub connection established.")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8d395168-698b-4bdb-9248-8120f0dc3bfb&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5bf6022bf9935ae320af8ec696d850074c5cf9cc32a4c243108d8a526f982545




Accessing as GigiSichinava

Initialized MLflow to track repo "GigiSichinava/ML-Assignment-2"

Repository GigiSichinava/ML-Assignment-2 initialized!

DagsHub connection established.


## Load Raw Test Data
The raw test data is loaded exactly as downloaded from Kaggle with no preprocessing applied. The Pipeline stored in the registry handles all transformations internally.

In [3]:
# Load raw test data without any preprocessing
# The pipeline handles all transformations internally
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
test_df = test_transaction.merge(test_identity, on='TransactionID', how='left')

transaction_ids = test_df['TransactionID'].copy()
test_df = test_df.drop(columns=['TransactionID'])

print(f"Raw test data shape: {test_df.shape}")


Raw test data shape: (506691, 432)


## Generate Predictions and Export Submission
The champion pipeline is loaded from the registry and applied directly to raw test data. We take the probability of the positive class (fraud) as the prediction score, which is what Kaggle requires for this competition.

In [4]:
# Load the champion pipeline directly from DagsHub Model Registry
# The best model across all architectures is registered as FraudDetection_Champion
model_uri = "models:/XGBoost_Pipeline/latest"
print(f"Loading model: {model_uri}")
pipeline = mlflow.sklearn.load_model(model_uri)

# Generate fraud probability scores
# We use predict_proba and take the positive class column (index 1)
predictions = pipeline.predict_proba(test_df)[:, 1]

# Build the submission file in the required Kaggle format
submission = pd.DataFrame({
    'TransactionID': transaction_ids,
    'isFraud': predictions
})
submission.to_csv('submission.csv', index=False)

print(f"Predictions generated: {len(predictions)} rows")
print(f"Sample fraud scores: {predictions[:5].round(4).tolist()}")
print(submission.head())
print("\nsubmission.csv is ready for Kaggle upload.")


Loading model: models:/XGBoost_Pipeline/latest


/tmp/ipykernel_57/3995306459.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_57/3995306459.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_57/3995306459.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
/tmp/ipykernel_57/3995306459.py:29: PerformanceWarning: DataF

Predictions generated: 506691 rows
Sample fraud scores: [0.002199999988079071, 0.004800000227987766, 0.002400000113993883, 0.00419999985024333, 0.016300000250339508]
   TransactionID   isFraud
0        3663549  0.002156
1        3663550  0.004849
2        3663551  0.002378
3        3663552  0.004175
4        3663553  0.016276

submission.csv is ready for Kaggle upload.
